In [1]:
import numpy as np

def to_embedding(text, embedding_dim=4, seed=42):
    np.random.seed(seed)    # to keep randomness parmanent

    tokens = text.lower().split()

    vocab = sorted(set(tokens))
    word_to_idx = {word: idx for idx, word in enumerate(vocab)}

    # Random embedding matrix
    embedding_matrix = np.random.randn(len(vocab), embedding_dim)

    # Convert sentence to embeddings
    sentence_embeddings = np.array([
        embedding_matrix[word_to_idx[word]]
        for word in tokens
    ])

    return sentence_embeddings, embedding_matrix, word_to_idx

In [2]:
# function to get one word's embedding
def get_embedding(word, embedding_matrix, vocab):
    return embedding_matrix[vocab[word.lower()]]

In [3]:
sentence = "The cat sat on the mat"

embeddings, embedding_matrix, vocab = to_embedding(sentence)

In [4]:
embeddings, vocab

(array([[-1.01283112,  0.31424733, -0.90802408, -1.4123037 ],
        [ 0.49671415, -0.1382643 ,  0.64768854,  1.52302986],
        [ 0.24196227, -1.91328024, -1.72491783, -0.56228753],
        [-0.46947439,  0.54256004, -0.46341769, -0.46572975],
        [-1.01283112,  0.31424733, -0.90802408, -1.4123037 ],
        [-0.23415337, -0.23413696,  1.57921282,  0.76743473]]),
 {'cat': 0, 'mat': 1, 'on': 2, 'sat': 3, 'the': 4})

In [5]:
# get cat's embedding from matrix
cat_embedding = get_embedding("cat", embedding_matrix, vocab)
the_embedding = get_embedding("the", embedding_matrix, vocab)
sat_embedding = get_embedding("sat", embedding_matrix, vocab)

print(cat_embedding)
print(the_embedding)
print(sat_embedding)

[ 0.49671415 -0.1382643   0.64768854  1.52302986]
[-1.01283112  0.31424733 -0.90802408 -1.4123037 ]
[ 0.24196227 -1.91328024 -1.72491783 -0.56228753]


In [6]:
def encode_position(word_embedding, pos):
  dmodel = len(word_embedding)
  final_encoding = []

  for i in range(dmodel // 2):
    omega = 1 / (10000 ** ((2 * i) / dmodel))

    # getting encodings on sin / cos functions
    pos_encoding0 = np.sin(omega * pos)
    pos_encoding1 = np.cos(omega * pos)

    # appending encodings
    final_encoding.extend([pos_encoding0, pos_encoding1])

  embedding = np.array(word_embedding)
  encoding = np.array(final_encoding)

  # add positional encoding with word's embedding
  pos_encoded_vector = embedding + encoding
  return encoding, pos_encoded_vector

In [7]:
cat_encoded_vector, cat_embedded_vector = encode_position(cat_embedding, 1)

In [8]:
cat_encoded_vector

array([0.84147098, 0.54030231, 0.00999983, 0.99995   ])

In [9]:
cat_embedded_vector

array([1.33818514, 0.402038  , 0.65768837, 2.52297986])

# 🎯 Conclusion

In this notebook, i built an intuitive and mathematical understanding of **Sinusoidal Positional Encoding**, one of the core ideas behind the original Transformer architecture. We explored why Transformers require positional information, how sinusoidal functions generate unique positional representations, and why multiple frequencies help encode both local and global positions. Most importantly, we derived the property

\[ PE(pos + k) = A_k \. PE(pos) \]

which explains how Transformers naturally learn **relative positional relationships** instead of memorizing absolute positions. Finally, we implemented positional encoding from scratch and combined it with word embeddings to create the final input embeddings fed into the Transformer.

> **Key Takeaway:** Positional encoding is much more than a formula, it's a mathematically elegant way of injecting sequence order while preserving relative distances between tokens, enabling self-attention to reason about word order without recurrence or convolution.

# Transformation Matrix
---
### Linear Transformation Property

One of the most important properties of **Sinusoidal Positional Encoding** is that the encoding at a position shifted by \(k\) can be obtained by applying a **fixed linear transformation** to the encoding at the original position.

For a single sine-cosine pair,

$$
PE(pos)=
\begin{bmatrix}
\sin(\omega\,pos) \\
\cos(\omega\,pos)
\end{bmatrix}
$$

where

$$
\omega=\frac{1}{10000^{2i/d_{\text{model}}}}
$$

Using the trigonometric identities,

$$
\sin(a+b)=\sin a\cos b+\cos a\sin b
$$

$$
\cos(a+b)=\cos a\cos b-\sin a\sin b
$$

we obtain,

$$
PE(pos+k)=A_k\,PE(pos)
$$

where the transformation matrix is

$$
A_k=
\begin{bmatrix}
\cos(\omega k) & \sin(\omega k)\\
-\sin(\omega k) & \cos(\omega k)
\end{bmatrix}
$$

Substituting the matrix,

$$
\begin{bmatrix}
\sin(\omega(pos+k))\\
\cos(\omega(pos+k))
\end{bmatrix}
=
\begin{bmatrix}
\cos(\omega k) & \sin(\omega k)\\
-\sin(\omega k) & \cos(\omega k)
\end{bmatrix}
\begin{bmatrix}
\sin(\omega pos)\\
\cos(\omega pos)
\end{bmatrix}
$$

### 💡 Key Insight

- The transformation matrix \(A_k\) depends **only on the relative distance** \(k\), not on the absolute position \(pos\).
- The same matrix transforms **Position 5 → 6**, **100 → 101**, and **1000 → 1001** because each shift has the same offset (\(k=1\)).
- This allows the Transformer to learn **relative word positions** (e.g., "next token", "two tokens before") instead of memorizing absolute positions.

In [10]:
def transform_pe(pos_encoding, k):
  dmodel = pos_encoding.shape[0]
  transformed = []

  for i in range(dmodel // 2):
    omega = 1 / (10000 ** ((2 * i) / dmodel))

    A_k = np.array([
          [ np.cos(omega * k),  np.sin(omega * k)],
          [-np.sin(omega * k),  np.cos(omega * k)]
    ])

    pe_pair = pos_encoding[2 * i : 2 * i + 2]
    transformed.extend(A_k @ pe_pair)

  return np.array(transformed)

In [11]:
# reach 'sat' position using 'cat' encoding with distance k = +1
trans_pe2 = transform_pe(cat_encoded_vector, 1)
sat_encoding, sat_final_vector = encode_position(sat_embedding, 2)

In [13]:
# compare transformed encoded vector to actual vector
np.allclose(trans_pe2, sat_encoding)

True

- Floating-point numbers (especially involving sin, cos, sqrt, matrix multiplications, etc.) → use np.allclose()